# <font color='steelblue'>Introducción a scikit-learn</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 02/06/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

In [ ]:
# @title Cargar configuración del cuaderno
!pip install gdown
!pip install --upgrade kagglehub
!pip install lightgbm xgboost
!pip install ucimlrepo

# Análisis numérico
import numpy as np
import pandas as pd
import math, random
import warnings
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format = 'retina'

import os, zipfile, gdown, kagglehub

# Funciones del curso
from urllib.request import urlretrieve
for fichero in ['preprocesar.py', 'evaluar_clasificadores.py', 'pca_funciones.py', 'auto_ML.py']:
    urlretrieve(f'https://raw.githubusercontent.com/ia4legos/MachineLearning/refs/heads/main/{fichero}', fichero)
from preprocesar import *
from evaluar_clasificadores import *
from pca_funciones import *
from auto_ML import *




# <font color="steelblue">1. Estudios de caso iniciales</font>

A continuación presentamos y cargamos los bancos de datos que vamos a utilizar en este cuaderno.

**stroke:** Este conjunto de datos se utiliza para predecir si un paciente tiene más o menos probabilidad de sufrir un ictus, en función de su género, edad, enfermedades y hábito de fumar. Cada fila proporciona información sobre un paciente. Las variables predictoras son `gender`, `age`, `hypertension`, `heart_disease`, `ever_married`, `work_type`, `Residence_type`, `avg_glucose_level`, `bmi` y `smoking_status`; la variable respuesta, categórica, es `stroke`, con valores `Yes` y `No`. El fichero incluye además una columna identificadora `id` que debe descartarse antes de modelar. Conviene tener en cuenta dos detalles: la variable `bmi` presenta valores perdidos (201 registros), y el conjunto está muy desequilibrado (solo en torno al 5 % de los pacientes sufren un ictus).

**alcohol:** Los datos de consumo de alcohol entre adolescentes se obtuvieron de una encuesta a alumnos de matemáticas y de lengua portuguesa de enseñanza secundaria. Además de las variables de consumo, se dispone de mucha información social, de género y de estudios. En este caso no hay variable respuesta (uso no supervisado). El fichero que cargamos, `student-mat.csv`, corresponde **únicamente al curso de matemáticas** (395 alumnos); los datos del curso de lengua portuguesa están en un fichero distinto.

Las variables consideradas son `school`, `sex`, `age`, `address`, `famsize`, `Pstatus`, `Medu`, `Fedu`, `Mjob`, `Fjob`, `reason`, `guardian`, `traveltime`, `studytime`, `failures`, `schoolsup`, `famsup`, `paid`, `activities`, `nursery`, `higher`, `internet`, `romantic`, `famrel`, `freetime`, `goout`, `Dalc`, `Walc`, `health`, `absences`, `G1`, `G2` y `G3`. Aquí `Dalc` y `Walc` recogen el consumo de alcohol entre semana y en fin de semana, y `G1`, `G2`, `G3` son las calificaciones de los tres periodos.

El código de lectura de los bancos de datos propuestos es:


In [ ]:
import pandas as pd

# CARGA DE LOS BANCOS DE DATOS EN EL CUADERNO

# Stroke Prediction Dataset
url = 'https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/stroke_ori.csv'
stroke = pd.read_csv(url)

# Student Alcohol Consumption (curso de matemáticas)
url = 'https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/student-mat.csv'
alcohol = pd.read_csv(url)

# <font color="steelblue">2. Preprocesado de datos</font>

Los algoritmos de aprendizaje necesitan datos **numéricos, completos y, en muchos casos, en escalas comparables**. El preprocesamiento es la etapa que prepara las variables para el modelado: imputa los valores perdidos, escala las variables numéricas y codifica las categóricas. En este apartado lo resolvemos con un conjunto reducido de herramientas de scikit-learn:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

Cada una cumple un papel: `train_test_split` separa los datos, `SimpleImputer` imputa, `StandardScaler` escala, `OneHotEncoder` codifica, `Pipeline` encadena pasos y `ColumnTransformer` aplica transformaciones distintas a columnas distintas.

> **Principio rector: evitar la fuga de información (*data leakage*).** Todo lo que el preprocesado *aprende* de los datos —la mediana con la que se imputa, la media y la desviación con las que se escala, las categorías que se codifican— debe calcularse **únicamente con la muestra de entrenamiento** y aplicarse después a la de test. Si la muestra de test interviene en esos cálculos, la evaluación resultará demasiado optimista. Por eso **dividimos antes de preprocesar**.

**Dividir antes de preprocesar: `train_test_split`**

El primer paso es separar predictoras ($X$) y objetivo ($y$) y, sobre todo, reservar la muestra de test **antes** de cualquier transformación:

In [ ]:
datos = stroke.copy()
target = 'stroke'

X = datos.drop(columns=target)
y = datos[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

`test_size` fija la proporción reservada para test, `random_state` garantiza la reproducibilidad y `stratify=y` mantiene las proporciones de clase en ambas particiones (recomendable en clasificación).

**Identificar el tipo de cada variable**

Como las variables numéricas y las categóricas requieren tratamientos distintos, separamos sus nombres:

In [ ]:
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(include=['object', 'category']).columns.tolist()

**Tratamiento de las variables numéricas: `SimpleImputer` + `StandardScaler`**

Para las numéricas encadenamos dos pasos con un `Pipeline`: primero imputamos los valores perdidos por la **mediana** (robusta frente a valores atípicos) y después estandarizamos con `StandardScaler`, que transforma cada variable a **media 0 y desviación 1**:

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

El escalado es importante para modelos sensibles a la escala o a las distancias (regresión logística, *k*-vecinos, SVM, redes neuronales); los modelos basados en árboles no lo necesitan, pero aplicarlo no los perjudica.

**Tratamiento de las variables categóricas: `SimpleImputer` + `OneHotEncoder`**

Para las categóricas, también con un `Pipeline`, imputamos por la **moda** (el valor más frecuente) y codificamos con `OneHotEncoder`, que crea una columna binaria por categoría:

In [ ]:
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
])

Dos opciones útiles de `OneHotEncoder`: `handle_unknown='ignore'` evita errores si en test aparece una categoría no vista en entrenamiento (la codifica como todo ceros), y `drop='first'` elimina una categoría de referencia para evitar la colinealidad (conveniente en modelos lineales; puede omitirse con modelos de árboles si se prefiere conservar todas las categorías).

**Combinar por tipo de columna: `ColumnTransformer`**

`ColumnTransformer` aplica cada *pipeline* al grupo de columnas correspondiente y une los resultados en una sola matriz:


In [ ]:
preprocesador = ColumnTransformer([
    ('num', num_pipe, num),
    ('cat', cat_pipe, cat),
])

**Ajustar con el entrenamiento y transformar ambas muestras**

Por último, **ajustamos el preprocesador solo con la muestra de entrenamiento** (`fit_transform`) y aplicamos esa misma transformación a la de test (`transform`), de modo que test se transforma con los parámetros aprendidos de train:

In [ ]:
X_train_prep = preprocesador.fit_transform(X_train)   # aprende y transforma
X_test_prep = preprocesador.transform(X_test)         # solo transforma

print("Train:", X_train_prep.shape, "| Test:", X_test_prep.shape)

Train: (4088, 17) | Test: (1022, 17)


> Esta distinción `fit_transform` (en train) frente a `transform` (en test) es la que materializa el principio de no fuga de información: la mediana, la media, la desviación y las categorías salen exclusivamente del entrenamiento.

**Integración con el modelo**

Una ventaja decisiva de trabajar con `Pipeline` es que el preprocesador puede **encadenarse con el propio modelo** en un único objeto:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo = Pipeline([
    ('prep', preprocesador),
    ('clf', RandomForestClassifier(random_state=42)),
])

modelo.fit(X_train, y_train)        # preprocesa y entrena en un paso
modelo.score(X_test, y_test)        # preprocesa test y evalúa (accuracy)

0.952054794520548

Así el preprocesado se reajusta automáticamente con los datos correctos en cada situación. Esto es especialmente importante en **validación cruzada**, donde el `Pipeline` garantiza que, en cada pliegue, la imputación, el escalado y la codificación se aprenden solo con los datos de entrenamiento de ese pliegue, evitando la fuga de información de forma transparente.

A partir del código detallado presentado generamos funciones genéricas que utilizaremos para el preprocesado de datos. Posteriormente las integraremos todas ellas en un fichero **.py** que cargaremos al inicio de cada cuaderno. En algunos casos será necesario volver al código detallado pero en la mayoría de casos podremos utilizar dichas funciones.




## Función de preprocesado

Comenzamos definiendo dos funciones para el preprocesado de un dataframe. En la primera que es la más general consideramos todas las tareas de preprocesamiento, mientras que en la segunda (que utilizaremos en los modelos que no necesitan estandarización de variables numéricas) solo se consideran las tareas de imputación y codificación de factores:

* Función 1 `preprocesar_datos`:
  *  En el proceso de imputación utilizamos la mediana para variables numéricas, y la clase más frecuente en las categóricas.
  * En el proceso de codificación utilizamos onehot encoder.
  * En el proceso de estandarización utilizamos `Standarscaler`.
* Función 2 `preprocesar_datos_se`:
  *  En el proceso de imputación utilizamos la mediana para varaibles numéricas, y la clase más frecuente en las categóricas.
  * En el proceso de codificación utilizamos onehot encoder.

Ambas funciones utilizan los mismos argumentos de entrada. LAs opciones de preprocesado se pueden mofificar decargando la función correspondiente y editándola adecuadamente.

In [ ]:
help(preprocesar_datos)

Help on function preprocesar_datos in module preprocesar:

preprocesar_datos(df, target=None, preprocessor=None)
    Preprocesa un DataFrame SIN fuga de información.
    - preprocessor=None  -> AJUSTA con `df` (train), transforma y devuelve (df_prep, preprocessor).
    - preprocessor dado   -> solo TRANSFORMA `df` (test) con lo aprendido del train.
    Devuelve SIEMPRE una tupla (df_preprocesado, preprocessor).



```python
def preprocesar_datos(df, target=None, preprocessor=None):
    """
    Preprocesa un DataFrame SIN fuga de información.
    - preprocessor=None  -> AJUSTA con `df` (train), transforma y devuelve (df_prep, preprocessor).
    - preprocessor dado   -> solo TRANSFORMA `df` (test) con lo aprendido del train.
    Devuelve SIEMPRE una tupla (df_preprocesado, preprocessor).
    """
    dfs = df.copy() if target is None else df.drop(columns=target)

    # Las booleanas se tratan como texto (consistente en train y test)
    bool_features = dfs.select_dtypes(include='bool').columns
    if len(bool_features):
        dfs[bool_features] = dfs[bool_features].astype(str)

    if preprocessor is None:
        # AJUSTE (con la muestra de entrenamiento)
        numeric_features = dfs.select_dtypes(include=np.number).columns.tolist()
        categorical_features = dfs.select_dtypes(include=['object', 'category']).columns.tolist()
        if not numeric_features and not categorical_features:
            raise ValueError("No hay variables numéricas ni categóricas que preprocesar.")
        transformers = []
        if numeric_features:
            transformers.append(('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())]), numeric_features))
        if categorical_features:
            transformers.append(('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first',
                                         sparse_output=False))]), categorical_features))
        preprocessor = ColumnTransformer(transformers=transformers)
        preprocessor.set_output(transform='pandas')
        df_prep = preprocessor.fit_transform(dfs)
    else:
        # TRANSFORMACIÓN (con test, usando lo aprendido del train)
        df_prep = preprocessor.transform(dfs)

    # Quitamos los prefijos 'num__' / 'cat__'
    df_prep.columns = [c.split('__', 1)[-1] for c in df_prep.columns]

    # Reincorporamos el target sin transformar
    if target is not None:
        df_prep = pd.concat([df_prep, df[target]], axis=1)

    return df_prep, preprocessor
```

In [ ]:
help(preprocesar_datos_se)

Help on function preprocesar_datos_se in module preprocesar:

preprocesar_datos_se(df, target=None, preprocessor=None)
    Preprocesa un DataFrame SIN fuga de información.
    Solo aplica: imputación de missings (numéricas y categóricas) y
    codificación One-Hot de las categóricas. NO estandariza las numéricas.
    - preprocessor=None  -> AJUSTA con `df` (train), transforma y devuelve (df_prep, preprocessor).
    - preprocessor dado   -> solo TRANSFORMA `df` (test) con lo aprendido del train.
    Devuelve SIEMPRE una tupla (df_preprocesado, preprocessor).



## Función para división de muestras

Definimos ahora una función para la divión del conjunto de datos en muestra de entrenamiento-validación y muestra de test. Por defecto la función esratifica por el traget en modelos supervisados para tratar con el desquilibrio de clases en la selección de muestras.

In [ ]:
help(split_sample)

Help on function split_sample in module preprocesar:

split_sample(df, target, size=0.2, semilla=42, estratificar=True)
    Divide un DataFrame en entrenamiento y test, opcionalmente estratificando por el target.
    Devuelve (strain, stest).



```python
def split_sample(df, target, size=0.2, semilla=42, estratificar=True):
    """
    Divide un DataFrame en entrenamiento y test, opcionalmente estratificando por el target.
    Devuelve (strain, stest).
    """
    if target not in df.columns:
        raise KeyError(f"La variable objetivo '{target}' no está en el DataFrame.")
    if not 0 < size < 1:
        raise ValueError(f"`size` debe estar entre 0 y 1 (recibido: {size}).")

    X = df.drop(columns=target)
    y = df[target]
    estrato = y if estratificar else None
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=size, random_state=semilla, stratify=estrato)
        print(f"{'Estratificando' if estratificar else 'División sin estratificar'} por '{target}'.")
    except ValueError:
        print(f"Aviso: no es posible estratificar por '{target}' "
              f"(hay clases con un solo caso o el target es continuo). Se divide sin estratificar.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=size, random_state=semilla)

    strain = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)
    stest = pd.concat([X_test, y_test], axis=1).reset_index(drop=True)
    print(f"  Entrenamiento: {len(strain)} muestras | Test: {len(stest)} muestras")
    return strain, stest
```

## Transformer de preprocesado

Definimos ahora el transformer de preprocesado utilizando las funciones definidas y lo aplicamos a los bancos de datos `stroke` y `alcohol`.

### Banco de datos `stroke`

In [ ]:
## Banco de datos stroke
target = 'stroke'
df = stroke.copy()

# 1) DIVIDIR primero (datos crudos)
strain_raw, stest_raw = split_sample(df, target, size=0.2, semilla=42, estratificar=True)
# 2) AJUSTAR el preprocesado SOLO con el entrenamiento (devuelve el preprocesador)
strain, prep = preprocesar_datos(strain_raw, target=target)
# 3) TRANSFORMAR el test con lo aprendido del entrenamiento
stest, _ = preprocesar_datos(stest_raw, target=target, preprocessor=prep)

Estratificando por 'stroke'.
  Entrenamiento: 4088 muestras | Test: 1022 muestras


Valoramos el conjunto de datos antes de preprocesado.

In [ ]:
# Stroke original y análisis descriptivo
print("Valores missing en stroke:", stroke.isnull().sum())
print("\nDescripción de datos:")
display(stroke.describe(include='all'))

Valores missing en stroke: id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Descripción de datos:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
count,5110.000000,5110,5110.000000,5110,5110,5110,5110,5110,5110.000000,4909.000000,5110,5110
unique,NaN,3,NaN,2,2,2,5,2,NaN,NaN,4,2
top,NaN,Female,NaN,No,No,Yes,Private,Urban,NaN,NaN,never smoked,No
freq,NaN,2994,NaN,4612,4834,3353,2925,2596,NaN,NaN,1892,4861
mean,36517.829354,NaN,43.226614,NaN,NaN,NaN,NaN,NaN,106.147677,28.893237,NaN,NaN
std,21161.721625,NaN,22.612647,NaN,NaN,NaN,NaN,NaN,45.283560,7.854067,NaN,NaN
min,67.000000,NaN,0.080000,NaN,NaN,NaN,NaN,NaN,55.120000,10.300000,NaN,NaN
25%,17741.250000,NaN,25.000000,NaN,NaN,NaN,NaN,NaN,77.245000,23.500000,NaN,NaN
50%,36932.000000,NaN,45.000000,NaN,NaN,NaN,NaN,NaN,91.885000,28.100000,NaN,NaN
75%,54682.000000,NaN,61.000000,NaN,NaN,NaN,NaN,NaN,114.090000,33.100000,NaN,NaN


Evaluamos los datos preprocesados:

In [ ]:
# Resumen muestra train
print("Valores missing en stroke:", strain.isnull().sum())
print("\nDescripción de datos:")
display(strain.describe(include='all'))

Valores missing en stroke: id                                0
age                               0
avg_glucose_level                 0
bmi                               0
gender_Male                       0
gender_Other                      0
hypertension_Yes                  0
heart_disease_Yes                 0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
stroke                            0
dtype: int64

Descripción de datos:


,id,age,avg_glucose_level,bmi,gender_Male,gender_Other,hypertension_Yes,heart_disease_Yes,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
count,4.088000e+03,4.088000e+03,4.088000e+03,4.088000e+03,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,4088
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3889
mean,-1.399185e-16,-1.564306e-17,3.702192e-16,-3.749990e-16,0.413894,0.000245,0.097114,0.054061,0.660470,0.003180,0.570450,0.163160,0.135519,0.506115,0.174658,0.367172,0.153131,NaN
std,1.000122e+00,1.000122e+00,1.000122e+00,1.000122e+00,0.492590,0.015640,0.296148,0.226165,0.473608,0.056309,0.495072,0.369557,0.342319,0.500024,0.379720,0.482093,0.360158,NaN
min,-1.717407e+00,-1.915251e+00,-1.131326e+00,-2.393908e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,-8.912351e-01,-7.680467e-01,-6.409288e-01,-6.548823e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,1.958164e-02,7.288256e-02,-3.175881e-01,-1.138522e-01,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,NaN
75%,8.589274e-01,7.810336e-01,1.741352e-01,5.044680e-01,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,NaN


In [ ]:
# Resumen muestra test
print("Valores missing en stroke:", stest.isnull().sum())
print("\nDescripción de datos:")
display(stest.describe(include='all'))

Valores missing en stroke: id                                0
age                               0
avg_glucose_level                 0
bmi                               0
gender_Male                       0
gender_Other                      0
hypertension_Yes                  0
heart_disease_Yes                 0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
stroke                            0
dtype: int64

Descripción de datos:


,id,age,avg_glucose_level,bmi,gender_Male,gender_Other,hypertension_Yes,heart_disease_Yes,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
count,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.0,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022.000000,1022
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,972
mean,-0.007025,-0.028032,-0.018726,-0.016572,0.413894,0.0,0.098826,0.053816,0.638943,0.008806,0.580235,0.148728,0.130137,0.515656,0.167319,0.382583,0.159491,NaN
std,0.982380,1.003798,1.003112,0.958593,0.492771,0.0,0.298574,0.225765,0.480542,0.093473,0.493762,0.355994,0.336619,0.500000,0.373443,0.486256,0.366313,NaN
min,-1.717878,-1.915251,-1.128233,-2.136274,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,-0.853987,-0.856566,-0.646729,-0.693527,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,0.013224,-0.015636,-0.325101,-0.113852,0.000000,0.0,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,NaN
75%,0.827271,0.781034,0.163473,0.488366,1.000000,0.0,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,NaN


### Banco de datos `alcohol`

En este caso no disponemos de target y solo realizamos la tarea de preprocesado sin división de muestras.

In [ ]:
alcohol_prep, prep = preprocesar_datos(alcohol, target=None)

In [ ]:
# Alcohol
# Este dataframe no tiene target definido
print("Valores missing en alcohol:", alcohol.isnull().sum())
print("\nDescripción de datos:")
display(alcohol.describe(include='all'))

Valores missing en alcohol: school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64

Descripción de datos:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395,395,395.000000,395,395,395,395.000000,395.000000,395,395,...,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
unique,2,2,NaN,2,2,2,NaN,NaN,5,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,GP,F,NaN,U,GT3,T,NaN,NaN,other,other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,349,208,NaN,307,281,354,NaN,NaN,141,217,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,16.696203,NaN,NaN,NaN,2.749367,2.521519,NaN,NaN,...,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,NaN,NaN,1.276043,NaN,NaN,NaN,1.094735,1.088201,NaN,NaN,...,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,NaN,NaN,15.000000,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,NaN,NaN,16.000000,NaN,NaN,NaN,2.000000,2.000000,NaN,NaN,...,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,NaN,NaN,17.000000,NaN,NaN,NaN,3.000000,2.000000,NaN,NaN,...,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,NaN,NaN,18.000000,NaN,NaN,NaN,4.000000,3.000000,NaN,NaN,...,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000


In [ ]:
# Datos preprocesado
print("Valores missing en alcohol:", alcohol_prep.isnull().sum())
print("\nDescripción de datos:")
display(alcohol_prep.describe(include='all'))

Valores missing en alcohol: age                  0
Medu                 0
Fedu                 0
traveltime           0
studytime            0
failures             0
famrel               0
freetime             0
goout                0
Dalc                 0
Walc                 0
health               0
absences             0
G1                   0
G2                   0
G3                   0
school_MS            0
sex_M                0
address_U            0
famsize_LE3          0
Pstatus_T            0
Mjob_health          0
Mjob_other           0
Mjob_services        0
Mjob_teacher         0
Fjob_health          0
Fjob_other           0
Fjob_services        0
Fjob_teacher         0
reason_home          0
reason_other         0
reason_reputation    0
guardian_mother      0
guardian_other       0
schoolsup_yes        0
famsup_yes           0
paid_yes             0
activities_yes       0
nursery_yes          0
higher_yes           0
internet_yes         0
romantic_yes         0
dtype:

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
count,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,3.950000e+02,...,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,1.223213e-15,5.396527e-17,-1.439074e-16,6.295948e-17,-2.113640e-16,-1.349132e-17,-1.394103e-16,1.079305e-16,-1.214219e-16,1.349132e-16,...,0.691139,0.081013,0.129114,0.612658,0.458228,0.508861,0.794937,0.949367,0.832911,0.334177
std,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,1.001268e+00,...,0.462610,0.273201,0.335751,0.487761,0.498884,0.500555,0.404260,0.219525,0.373528,0.472300
min,-1.330954e+00,-2.514630e+00,-2.320084e+00,-6.432495e-01,-1.235351e+00,-4.499436e-01,-3.287804e+00,-2.240828e+00,-1.896683e+00,-5.406987e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-5.462869e-01,-6.853872e-01,-4.798568e-01,-6.432495e-01,-1.235351e+00,-4.499436e-01,6.219406e-02,-2.360102e-01,-9.972953e-01,-5.406987e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
50%,2.383798e-01,2.292342e-01,-4.798568e-01,-6.432495e-01,-4.228585e-02,-4.499436e-01,6.219406e-02,-2.360102e-01,-9.790798e-02,-5.406987e-01,...,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
75%,1.023046e+00,1.143856e+00,4.402569e-01,7.922508e-01,-4.228585e-02,-4.499436e-01,1.178860e+00,7.663987e-01,8.014793e-01,5.833854e-01,...,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,4.161713e+00,1.143856e+00,1.360371e+00,3.663251e+00,2.343844e+00,3.589323e+00,1.178860e+00,1.768808e+00,1.700867e+00,3.955638e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Para que resulte más fácil el uso de estas funciones se han guardado en el fichero `preprocesar.py` que es accesible en la configuración del modelo. Es recomendable pegar dicho código al inicio de los cuadernos para tener acceso a dichas funciones.

# <font color="steelblue">3. Entrenamiento y comparación de modelos de aprendizaje</font>

En un problema de **clasificación** la variable objetivo es categórica: el modelo debe aprender a asignar cada observación a una de un conjunto finito de clases (por ejemplo, `Yes`/`No` en `stroke`, o las especies de Iris). En este apartado desarrollamos qué significa entrenar un clasificador, cómo estimar su capacidad real y cómo comparar varios modelos para elegir el más adecuado.



## Qué significa entrenar un clasificador

Entrenar (o *ajustar*) un modelo de clasificación consiste en estimar, a partir de un conjunto de datos etiquetados, una **función de decisión** que relacione las predictoras $X$ con la clase $y$. Internamente, el algoritmo busca los parámetros que minimizan una función de pérdida (por ejemplo, la *entropía cruzada*) sobre los datos de entrenamiento.

El objetivo no es memorizar los datos de entrenamiento, sino **generalizar**: acertar sobre datos nuevos no vistos. De ahí surge el compromiso central del aprendizaje:

- **Sobreajuste (*overfitting*):** el modelo es demasiado complejo, se ajusta al ruido del entrenamiento y rinde mal con datos nuevos. Tiene baja *varianza* explicada pero alta sensibilidad a la muestra concreta.
- **Subajuste (*underfitting*):** el modelo es demasiado simple, no captura la estructura de los datos y rinde mal incluso en entrenamiento.

Este equilibrio se conoce como el **compromiso sesgo–varianza** (*bias–variance tradeoff*): los modelos simples tienen mucho sesgo y poca varianza; los complejos, poco sesgo y mucha varianza. El buen rendimiento se obtiene en un punto intermedio, controlando la complejidad del modelo (sus **hiperparámetros**).



## Principales familias de modelos de clasificación

No existe un modelo universalmente mejor (principio de *no free lunch*): el más adecuado depende del problema, los datos y los objetivos. Por eso conviene conocer las familias principales y compararlas empíricamente.

| Modelo | Idea básica | Notas |
|---|---|---|
| **Regresión logística** | Estima la probabilidad de cada clase mediante una frontera lineal | Simple, interpretable; requiere escalado; base de comparación habitual |
| **Análisis discriminante (LDA/QDA)** | Asume distribuciones gaussianas por clase y deriva fronteras lineales (LDA) o cuadráticas (QDA) | Eficiente con pocos datos si se cumplen los supuestos |
| **k-vecinos más cercanos (k-NN)** | Clasifica según la clase mayoritaria entre los $k$ vecinos más próximos | Sin entrenamiento explícito; muy sensible a la escala y a la dimensionalidad |
| **Naive Bayes** | Aplica el teorema de Bayes suponiendo independencia entre predictoras | Muy rápido; útil en texto; el supuesto de independencia rara vez se cumple |
| **Árbol de decisión** | Divide el espacio en regiones mediante reglas sucesivas | Interpretable; tiende a sobreajustar si no se poda |
| **Random forest** | Promedio de muchos árboles entrenados sobre muestras y variables aleatorias | Robusto, poco sensible a la escala; buen rendimiento por defecto |
| **Gradient boosting** (XGBoost, LightGBM…) | Árboles añadidos secuencialmente, corrigiendo los errores anteriores | Suele dar el mejor rendimiento en datos tabulares; requiere ajuste |
| **Máquinas de vectores soporte (SVM)** | Busca el hiperplano de máximo margen entre clases (con *kernels* para fronteras no lineales) | Potente; requiere escalado; costoso con muchos datos |

Conviene recordar de apartados anteriores que algunos modelos (regresión logística, k-NN, SVM, redes) son **sensibles a la escala** y requieren estandarización, mientras que los basados en árboles no la necesitan.

Para facilitar la tarea de entrenamiento y comparación de mútiples modelos de clasificación se presentan dos funciones que nos ayudan en dicha tarea. De momento, solo mostramos las funciones ya que las iremos utilizando a lo largo del resto de cuadernos.

### Entrenamiento modelos de clasificación binaria

Comenzamos con la función apra las tareas de classificación binaria, es decir, el target sólo tiene dos posibles respuestas.

La base de la evaluación en clasificación binaria es la **matriz de confusión**, que cruza las clases reales con las predichas. A partir de ella se definen las métricas principales:

- **Exactitud (*accuracy*)**
- **Precisión (*precision*)**
- **Sensibilidad o exhaustividad (*recall*, TPR)**
- **Especificidad (TNR)**
- **F1**
- **Exactitud balanceada**

Existe siempre un **compromiso entre precisión y sensibilidad** que se regula con el **umbral de decisión** aplicado a la probabilidad predicha (por defecto 0,5). Dos herramientas analizan ese compromiso de forma global, sin fijar un umbral:

- **Curva ROC y AUC.** Representa la sensibilidad (TPR) frente a la tasa de falsos positivos (FPR) al variar el umbral; el **área bajo la curva (AUC)** resume la capacidad de discriminación (0,5 = azar, 1 = perfecto).
- **Curva precisión–sensibilidad (PR) y su área.** Más informativa que la ROC cuando el desequilibrio es fuerte, porque se centra en la clase minoritaria.

Otras métricas útiles son el **coeficiente kappa de Cohen** (acuerdo corregido por azar), el **coeficiente de correlación de Matthews (MCC)** (robusto ante el desequilibrio) y la **pérdida logarítmica (*log-loss*)**, que penaliza las probabilidades mal calibradas.

Para el análisis comprativo en modelo de clasificación binaria se ha definido una función que permite dicha comparación y que se carga en la configuración del cuaderno en el fichero `evaluar_clasificadores.py`

In [ ]:
help(comparar_clasificador_2cls)

Help on function comparar_clasificador_2cls in module evaluar_clasificadores:

comparar_clasificador_2cls(strain, target, sizeval=0.3, semilla=42, models_to_train=None, pos_label=None)
    Entrena varios modelos de clasificación binaria y devuelve sus métricas.

    Args:
        strain (pd.DataFrame): conjunto de entrenamiento (predictoras ya
            preprocesadas a numérico + columna objetivo).
        target (str): nombre de la columna objetivo (debe tener 2 clases).
        sizeval (float): fracción del conjunto reservada para validación.
        semilla (int): semilla de aleatorización (reproducibilidad).
        models_to_train (list, opcional): nombres de modelos a entrenar; si es
            None se entrenan todos.
        pos_label (opcional): clase considerada "positiva". Si es None, se
            elige automáticamente (1 si las clases son 0/1; 'Yes' si existe;
            en otro caso, la clase minoritaria).

    Returns:
        pd.DataFrame: métricas por modelo (Accur

In [ ]:
# -------------------------- Ejemplo de uso --------------------------
tabla = comparar_clasificador_2cls(strain, target='stroke', sizeval=0.3, semilla=42)
tabla.sort_values('Balanced_Accuracy', ascending=False)

Estratificando por 'stroke'.
  Entrenamiento: 2861 muestras | Test: 1227 muestras
Clase positiva: 'Yes'
Entrenando lr...
Entrenando ridge...
Entrenando lda...
Entrenando qda...
Entrenando nb...
Entrenando knn...
Entrenando svc...
Entrenando rbf...
Entrenando dt...
Entrenando rf...
Entrenando extra...
Entrenando bagging...
Entrenando ada...
Entrenando gbc...
Entrenando hgb...
Entrenando lightgbm...
Entrenando xgboost...


,Accuracy,Balanced_Accuracy,Precision,Recall,Specificity,F1,AUC
Algorithm,,,,,,,
qda,0.203749,0.581405,0.057859,1.000000,0.162811,0.109389,0.627014
nb,0.189079,0.573693,0.056872,1.000000,0.147386,0.107623,0.838318
xgboost,0.942135,0.526907,0.210526,0.066667,0.987147,0.101266,0.829920
dt,0.908720,0.517245,0.080645,0.083333,0.951157,0.081967,0.517245
lda,0.947840,0.514096,0.250000,0.033333,0.994859,0.058824,0.847058
gbc,0.947025,0.513668,0.222222,0.033333,0.994002,0.057971,0.851657
hgb,0.945395,0.512811,0.181818,0.033333,0.992288,0.056338,0.837632
knn,0.948655,0.506620,0.200000,0.016667,0.996572,0.030769,0.637425
extra,0.947025,0.505763,0.142857,0.016667,0.994859,0.029851,0.775050


### Entrenamiento modelos de clasificación múltiple

En problemas **multiclase**, las métricas binarias se extienden promediando por clases: *macro* (media simple entre clases, trata a todas por igual), *weighted* (ponderada por el tamaño de cada clase) y *micro* (agrega los conteos globales).

**Qué métrica elegir** depende del problema y del coste de cada tipo de error. En diagnóstico médico, por ejemplo, un falso negativo (no detectar un enfermo) suele ser mucho más grave que un falso positivo, por lo que se prioriza la **sensibilidad**; en otros contextos importa más la **precisión**. La *accuracy* rara vez es suficiente por sí sola. En esta situación hemos definido una nueva función apra esta comparación.

In [ ]:
help(comparar_clasificador_multicls)

Help on function comparar_clasificador_multicls in module evaluar_clasificadores:

comparar_clasificador_multicls(strain, target, sizeval=0.3, semilla=42, models_to_train=None)
    Entrena varios modelos de clasificación multiclase y devuelve sus métricas.

    Args:
        strain (pd.DataFrame): conjunto de entrenamiento (predictoras ya
            preprocesadas a numérico + columna objetivo).
        target (str): nombre de la columna objetivo.
        sizeval (float): fracción del conjunto reservada para validación.
        semilla (int): semilla de aleatorización (reproducibilidad).
        models_to_train (list, opcional): nombres de modelos a entrenar; si es
            None se entrenan todos.

    Returns:
        pd.DataFrame: métricas por modelo (Accuracy, Balanced_Accuracy,
            Precision, Recall, F1, AUC), indexado por algoritmo. Precisión,
            sensibilidad, F1 y AUC se promedian con `average='weighted'`
            (la AUC en modo One-vs-Rest).



In [ ]:
# -------------------------- Ejemplo de uso --------------------------
# tabla = comparar_clasificador_multicls(strain, target='quality', sizeval=0.3, semilla=42)
# tabla.sort_values('Balanced_Accuracy', ascending=False)

### Comparación y selección de modelos

Comparar modelos de forma honesta requiere un procedimiento cuidadoso:

1. **Misma métrica y mismas particiones.** Se evalúan todos los modelos candidatos con **validación cruzada** sobre el conjunto de entrenamiento, usando la misma métrica (acorde al problema) y, a ser posible, los mismos pliegues, para que la comparación sea justa.
2. **Ajuste de hiperparámetros.** Cada modelo se compara *en su mejor versión*. Los hiperparámetros se optimizan mediante **búsqueda en cuadrícula** (*grid search*) o **búsqueda aleatoria** (*random search*), siempre con validación cruzada sobre el entrenamiento.
3. **Evaluación final en el conjunto de prueba.** Una vez elegido el modelo y sus hiperparámetros, se entrena con todo el conjunto de entrenamiento y se evalúa **una sola vez** sobre el conjunto de prueba. Este resultado es la estimación honesta de su generalización.
4. **Comparación estadística (opcional).** Para decidir si la diferencia entre dos modelos es real o fruto del azar, pueden usarse pruebas pareadas sobre los resultados de los pliegues de la validación cruzada, o la **prueba de McNemar** para comparar dos clasificadores sobre el mismo conjunto de prueba.

Más allá del rendimiento numérico, la elección final también atiende a criterios prácticos: la **interpretabilidad** (clave en ámbitos como el sanitario o el financiero), el **coste computacional** de entrenamiento y predicción, la **calibración** de las probabilidades, la **robustez** ante datos ruidosos y la facilidad de mantenimiento del modelo.

>**Nota.** En cuadernos posteriores iremos viendo como entrenar de forma individual cada clasificador, como optimizar sus hiperparámetros, y que tipo de información numérica y/o gráfica podemos obtener que nos ayude a la interpretación del modelo.

# <font color="steelblue">4. Evaluación del modelo</font>

Como se comentó en el cuaderno anterior, en un modelo de aprendizaje automático los datos suelen dividirse en tres muestras:

- **entrenamiento**, para ajustar el modelo;
- **validación**, para evaluar su rendimiento durante el entrenamiento y guiar las mejoras;
- **test**, para evaluar el rendimiento final y la bondad del modelo.

Sin embargo, esta partición en tres subconjuntos tiene el inconveniente de que reduce el tamaño de la muestra de entrenamiento, lo que puede conducir a un ajuste deficiente. Por ese motivo, en adelante trabajaremos con **solo dos subconjuntos**:

- la **muestra de entrenamiento**, con la que resolveremos el entrenamiento y el reajuste óptimo del modelo;
- la **muestra test**, con la que resolveremos la evaluación final.

Conviene diferenciar dos etapas posteriores al entrenamiento que, aunque a menudo se mezclan y confunden en la literatura, responden a preguntas distintas:

- **Evaluar el modelo:** medir, mediante métricas, la eficiencia y calidad del modelo ajustado, tanto en la muestra de entrenamiento como en la de test. Permite saber cómo de bien predice los datos y detectar **sobreajuste** (cuando predice claramente mejor los datos de entrenamiento que los de test).
- **Validar el modelo:** verificar la **consistencia** del ajuste, es decir, su independencia de la muestra aleatoria concreta utilizada para entrenar y de su tamaño.

Dado que el aprtado de evalaución lo hemos detallado en el punto anterior, pasamos a comentar el proceso de validación.

## Validación del modelo

El propósito de la validación es comprobar si el modelo ajustado es **consistente** al proporcionar resultados. En modelos de *Machine Learning*, planteamos esa consistencia en dos términos:

- si influye en los resultados la **elección aleatoria** de la muestra de entrenamiento;
- si influye en los resultados el **tamaño** de la muestra de entrenamiento.

Si validáramos con una muestra independiente de las de entrenamiento y test (una muestra de validación), evitaríamos el sobreajuste, pero reduciríamos el tamaño del entrenamiento y, posiblemente, empeoraríamos la solución. Por eso se plantea la **validación cruzada**, que permite mantener solo dos muestras —entrenamiento y test— y resolver con la primera tanto el entrenamiento como la parte de la validación relativa a *cómo influye la elección de la muestra*, dejando la segunda para la evaluación final.

La segunda cuestión, *cómo influye el tamaño de la muestra de entrenamiento*, se estudia con la **curva de aprendizaje**, buscando estabilizar el error a partir de un tamaño dado.

Describimos ambos procedimientos a continuación. Veremos, no obstante, que este procedimiento común de validación para todos los modelos de *Machine Learning* se complementa, en modelos específicos, con la verificación de las hipótesis de partida que esos modelos requieren para ser válidos.

### Validación cruzada

La **validación cruzada** (*cross-validation* o CV) permite trabajar exclusivamente con la muestra de entrenamiento para entrenar y validar, reservando la muestra test para evaluar el ajuste.

La aproximación básica, llamada **k-fold CV**, divide el conjunto de entrenamiento en *k* subconjuntos (*folds*) y repite, en un bucle, el siguiente proceso para cada uno de ellos:

1. se entrena un modelo usando los *k − 1* subconjuntos restantes como datos de entrenamiento;
2. el modelo resultante se evalúa sobre el subconjunto retenido, obteniendo una medida de rendimiento.

La medida que se reporta como resultado del procedimiento es el **promedio** de los rendimientos calculados en el bucle, acompañado de su **dispersión**. Si el ajuste es consistente, todas las medidas resultarán similares y la dispersión será pequeña. Una dispersión grande —rendimientos muy distintos entre subconjuntos— delata un problema de **inconsistencia** que invalidaría el ajuste.

Esta aproximación puede ser computacionalmente costosa, pero no "gasta" tantos datos como apartar un subconjunto de validación independiente, por lo que se recomienda especialmente cuando el número de muestras no es muy grande. El procedimiento de k-fold CV se ilustra en la siguiente figura.

Existen variantes habituales según el problema:

- **k-fold estratificado:** mantiene en cada *fold* la proporción de clases del conjunto completo. Es la opción recomendada en **clasificación**, sobre todo con clases desequilibradas. El procedimiento de *k-fold CV* se ilustra en la siguiente figura.
<center><small><img src=https://raw.githubusercontent.com/ia4legos/MachineLearning/main/images/grid_search_cross_validation.png></small></center>

- **k-fold repetido:** repite el proceso con varias particiones aleatorias y promedia, para una estimación aún más estable.
- **Leave-One-Out (LOO):** caso extremo en el que *k* es igual al número de muestras; muy costoso, reservado para conjuntos muy pequeños.

### Curva de aprendizaje

La **curva de aprendizaje** responde a la segunda pregunta de la validación: *cómo influye el tamaño de la muestra de entrenamiento* en el rendimiento. Para construirla se entrena el modelo con subconjuntos de entrenamiento de tamaño creciente y, para cada tamaño, se representa el rendimiento (o el error) tanto en **entrenamiento** como en **validación** (estimado por validación cruzada). El resultado son dos curvas en función del número de muestras.

Su lectura es muy informativa sobre el comportamiento del modelo:

- **Las dos curvas convergen y el error se estabiliza** a partir de cierto tamaño: el modelo es consistente respecto al tamaño muestral y añadir más datos apenas aportaría mejora. Es la situación deseable.
- **Hay una brecha amplia entre ambas curvas** (buen rendimiento en entrenamiento, peor en validación): indica **sobreajuste** (alta varianza). En este caso, **conseguir más datos** suele ayudar a cerrar la brecha.
- **Ambas curvas convergen pero en un rendimiento pobre**: indica **subajuste** (alto sesgo). Aquí más datos no ayudarán; conviene un modelo más complejo o mejores variables.

El objetivo, por tanto, es comprobar que el error se **estabiliza** a partir de un tamaño determinado, lo que confirma que el ajuste no depende críticamente del tamaño de la muestra empleada.

# <font color="steelblue">5. Equilibrado de muestras</font>

Cuando las categorías de la variable objetivo están muy **desequilibradas** —los tamaños muestrales de las clases son muy distintos—, la capacidad de aprendizaje de los modelos se ve mermada. El motivo es que la mayoría de algoritmos optimizan el acierto global y, ante un fuerte desequilibrio, encuentran que la estrategia más "rentable" es predecir casi siempre la clase mayoritaria. El resultado es un modelo con una *accuracy* engañosamente alta pero incapaz de detectar la clase minoritaria, que suele ser justo la de interés (un fraude, una enfermedad, un fallo).

Un ejemplo concreto: en el conjunto `stroke`, solo el ~5 % de los pacientes sufren un ictus. Un modelo sin tratar este desequilibrio puede alcanzar un AUC razonable y, sin embargo, **detectar apenas el 2 % de los ictus reales**. Tras equilibrar, el mismo modelo llega a detectar el ~80 %. Por eso el equilibrado, unido a una elección adecuada de métricas, es una etapa clave.

> **Principio fundamental:** el equilibrado se aplica **únicamente al conjunto de entrenamiento**, nunca al de prueba. El conjunto de prueba debe conservar la proporción real de clases para que la evaluación refleje el comportamiento del modelo en producción. Además, si se usa validación cruzada, el remuestreo debe rehacerse **dentro de cada pliegue**; la forma segura de garantizarlo es integrar el remuestreo en un `Pipeline` de la librería *imbalanced-learn* (ver más abajo).

## Familias de técnicas

Existen tres grandes enfoques, que pueden combinarse:

1. **Nivel de datos (remuestreo).** Modifican el conjunto de entrenamiento para equilibrar las clases:
   - *Sobremuestreo* (aumentar la minoritaria): sobremuestreo aleatorio, **SMOTE** y sus variantes (Borderline-SMOTE, ADASYN, SMOTENC).
   - *Submuestreo* (reducir la mayoritaria): submuestreo aleatorio, **NearMiss**, Tomek Links, *Edited Nearest Neighbours* (ENN).
   - *Híbridos* (combinan ambos): SMOTEENN, SMOTETomek.
2. **Nivel de algoritmo (aprendizaje sensible al coste).** No tocan los datos, sino que penalizan más los errores en la clase minoritaria mediante pesos: el parámetro `class_weight='balanced'` (disponible en regresión logística, SVM, *random forest*, etc.) o `sample_weight`.
3. **Ajuste del umbral de decisión.** En clasificación binaria, en lugar del umbral por defecto de 0,5 sobre la probabilidad, se elige un umbral que mejore el equilibrio entre sensibilidad y precisión (por ejemplo, maximizando el F1 o a partir de la curva *precision-recall*).

A continuación detallamos los dos métodos clásicos basados en vecinos cercanos (SMOTE y NearMiss) y luego ampliamos con el resto de opciones.

## SMOTE

SMOTE (*Synthetic Minority Oversampling Technique*) es uno de los métodos de **sobremuestreo** más utilizados. En lugar de limitarse a replicar ejemplos de la clase minoritaria (lo que favorece el sobreajuste), **sintetiza** nuevos ejemplos por interpolación lineal entre instancias minoritarias próximas. El algoritmo funciona así:

- **Paso 1:** dado el conjunto de la clase minoritaria $A$, para cada $x \in A$ se obtienen sus $k$ vecinos más cercanos calculando la distancia euclídea entre $x$ y las demás muestras de $A$.
- **Paso 2:** la tasa de muestreo $N$ se fija según el grado de desequilibrio. Para cada $x \in A$ se seleccionan aleatoriamente $N$ de sus $k$ vecinos ($x_1, x_2, \dots, x_N$), formando el conjunto $A_1$.
- **Paso 3:** para cada $x_k \in A_1$ se genera un nuevo ejemplo mediante
$$x' = x + \text{rand}(0,1)\cdot |x - x_k|,$$
donde $\text{rand}(0,1)$ es un número aleatorio entre 0 y 1. El nuevo punto $x'$ se sitúa, por tanto, en el segmento que une $x$ con su vecino.

**Variantes habituales:**
- **Borderline-SMOTE:** genera ejemplos sintéticos solo cerca de la frontera entre clases, que es donde el clasificador más se equivoca.
- **ADASYN** (*Adaptive Synthetic Sampling*): genera más ejemplos sintéticos en las zonas donde la clase minoritaria es más difícil de aprender (más rodeada de mayoritaria).
- **SMOTENC / SMOTEN:** versiones que admiten variables **categóricas** (SMOTE estándar solo trabaja con variables numéricas).

**Limitaciones:** al interpolar, SMOTE puede crear ejemplos en zonas de solapamiento entre clases (ruido), y no funciona directamente con variables categóricas ni con valores perdidos, por lo que los datos deben preprocesarse antes (imputación + codificación + escalado).

## NearMiss

NearMiss es un método de **submuestreo** que equilibra eliminando ejemplos de la clase mayoritaria. La idea: cuando instancias de clases distintas están muy próximas, se eliminan las mayoritarias cercanas para aumentar la separación entre clases y facilitar la clasificación. Para evitar la pérdida de información típica del submuestreo aleatorio, se apoya en los vecinos más cercanos. Su funcionamiento básico es:

- **Paso 1:** el método calcula las distancias entre todos los registros de la clase mayoritaria y los de la clase minoritaria (se submuestrea la mayoritaria).
- **Paso 2:** se seleccionan los $n$ registros de la clase mayoritaria con las distancias más pequeñas respecto a los de la minoritaria.
- **Paso 3:** si hay $k$ registros en la clase minoritaria, el método dará como resultado $k \times n$ registros de la clase mayoritaria.

**Versiones de NearMiss:**
- **NearMiss-1:** conserva las muestras mayoritarias cuya distancia media a las *k* minoritarias más cercanas es mínima.
- **NearMiss-2:** conserva las mayoritarias cuya distancia media a las *k* minoritarias más lejanas es mínima.
- **NearMiss-3:** para cada muestra minoritaria conserva sus vecinas mayoritarias; tiende a generar fronteras más limpias.

## Métodos adicionales

Más allá de SMOTE y NearMiss, conviene conocer el resto de opciones del ecosistema *imbalanced-learn*:

| Método | Tipo | Idea principal |
|---|---|---|
| **RandomOverSampler** | Sobremuestreo | Replica al azar ejemplos minoritarios (simple; riesgo de sobreajuste). |
| **RandomUnderSampler** | Submuestreo | Elimina al azar ejemplos mayoritarios (simple; pierde información). |
| **Borderline-SMOTE / ADASYN** | Sobremuestreo | Variantes de SMOTE centradas en la frontera o en zonas difíciles. |
| **Tomek Links** | Limpieza | Elimina pares de clases distintas que son vecinos mutuos (limpia la frontera). |
| **ENN** (*Edited Nearest Neighbours*) | Limpieza | Elimina ejemplos mal clasificados por sus vecinos. |
| **SMOTEENN** | Híbrido | SMOTE + limpieza con ENN. |
| **SMOTETomek** | Híbrido | SMOTE + eliminación de Tomek Links. |
| **`class_weight='balanced'`** | Nivel algoritmo | No toca los datos: pondera más los errores en la minoritaria. |

> **Consejo:** prueba primero `class_weight='balanced'`. Es la opción más simple, no genera datos sintéticos ni descarta información, y a menudo iguala el rendimiento del remuestreo.

## Métricas adecuadas

Con clases desequilibradas, la *accuracy* es engañosa. Conviene fijarse en:

- **Recall (sensibilidad)** de la clase minoritaria: de los positivos reales, cuántos detecta.
- **Precisión** de la clase minoritaria: de lo que predice como positivo, cuánto acierta.
- **F1:** media armónica de precisión y recall.
- **Balanced accuracy:** media de la sensibilidad de cada clase.
- **ROC-AUC** y, especialmente con desequilibrio fuerte, **PR-AUC** (área bajo la curva *precision-recall*).
- La **matriz de confusión**, para ver el reparto exacto de aciertos y errores.

## Práctica en Python

Usaremos la librería imbalanced-learn (imblearn), que se integra con scikit-learn.

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, NearMiss, TomekLinks
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

**Diagnóstico del desequilibrio**


In [ ]:
stroke = pd.read_csv(
    'https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/stroke_ori.csv'
).drop(columns='id')

X = stroke.drop(columns='stroke')
y = (stroke['stroke'] == 'Yes').astype(int)   # 1 = ictus, 0 = no ictus

print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

stroke
0    4861
1     249
Name: count, dtype: int64
stroke
0    0.951
1    0.049
Name: proportion, dtype: float64


**División y preprocesado (equilibrar SOLO el train)**


In [ ]:
# 1) Primero dividimos, para no contaminar el test con datos sintéticos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 2) SMOTE/NearMiss requieren datos numéricos sin nulos: preprocesamos
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(include=['object', 'category']).columns.tolist()
preprocesado = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('esc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first',
                                            sparse_output=False))]), cat),
])

**Aplicación de cada técnica y conteo resultante**

A modo ilustrativo, aplicamos cada técnica sobre el train ya preprocesado y mostramos los conteos por clase:

In [ ]:
X_train_p = preprocesado.fit_transform(X_train)

tecnicas = {
    'RandomOverSampler': RandomOverSampler(random_state=42),
    'SMOTE': SMOTE(random_state=42),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'RandomUnderSampler': RandomUnderSampler(random_state=42),
    'NearMiss (v1)': NearMiss(version=1),
    'TomekLinks': TomekLinks(),
    'SMOTEENN': SMOTEENN(random_state=42),
    'SMOTETomek': SMOTETomek(random_state=42),
}

for nombre, sampler in tecnicas.items():
    X_res, y_res = sampler.fit_resample(X_train_p, y_train)
    print(f"{nombre:20s} -> {pd.Series(y_res).value_counts().to_dict()}")

RandomOverSampler    -> {0: 3889, 1: 3889}
SMOTE                -> {0: 3889, 1: 3889}
BorderlineSMOTE      -> {0: 3889, 1: 3889}
ADASYN               -> {1: 3909, 0: 3889}
RandomUnderSampler   -> {0: 199, 1: 199}
NearMiss (v1)        -> {0: 199, 1: 199}
TomekLinks           -> {0: 3794, 1: 199}
SMOTEENN             -> {1: 3753, 0: 3177}
SMOTETomek           -> {0: 3879, 1: 3879}


Se observará que los métodos de sobremuestreo igualan las clases subiendo la minoritaria (p. ej. {0: 3889, 1: 3889}), los de submuestreo las igualan bajando la mayoritaria (p. ej. {0: 199, 1: 199}), y los de limpieza (TomekLinks, ENN) solo eliminan unos pocos ejemplos fronterizos.

El flujo correcto: remuestreo dentro de un Pipeline
Para evitar la fuga de información, el remuestreo debe aplicarse solo al entrenamiento. La forma robusta de garantizarlo es usar el Pipeline de imblearn, que aplica el remuestreo durante el ajuste (`fit`) pero no al predecir sobre datos nuevos. Comparamos el modelo base, el ajuste sensible al coste (`class_weight`) y SMOTE. Incorporamos las funciones de preprocesado y división de muestras quehemos implementado antes:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score)

target = 'stroke'   # ajusta al nombre de tu objetivo
df = stroke.copy()

# 1) Preprocesado (predictoras numéricas + target al final, sin transformar)
df_preprocesado, preprocessor = preprocesar_datos(df, target=target)

# 2) División (estratificando por el target)
strain, stest = split_sample(df_preprocesado, target, size=0.2, semilla=42, estratificar=True)

# 3) Separar predictoras y objetivo. OJO: preprocesar_datos deja el target SIN codificar,
#    así que lo pasamos a 0/1 (necesario para recall/F1/AUC binarias y para XGBoost)
X_train = strain.drop(columns=target);  y_train = (strain[target] == 'Yes').astype(int)
X_test  = stest.drop(columns=target);   y_test  = (stest[target]  == 'Yes').astype(int)

def evaluar(nombre, modelo):
    modelo.fit(X_train, y_train)
    pred  = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    print(f"{nombre:14s} recall={recall_score(y_test, pred):.3f}  "
          f"F1={f1_score(y_test, pred):.3f}  "
          f"bal_acc={balanced_accuracy_score(y_test, pred):.3f}  "
          f"AUC={roc_auc_score(y_test, proba):.3f}")

# Datos YA preprocesados → los modelos van "desnudos" (sin ColumnTransformer)
evaluar("Baseline",     LogisticRegression(max_iter=1000))
evaluar("class_weight", LogisticRegression(max_iter=1000, class_weight='balanced'))
evaluar("SMOTE",        ImbPipeline([('smote', SMOTE(random_state=42)),
                                     ('clf',   LogisticRegression(max_iter=1000))]))

Estratificando por 'stroke'.
  Entrenamiento: 4088 muestras | Test: 1022 muestras
Baseline       recall=0.020  F1=0.039  bal_acc=0.510  AUC=0.842
class_weight   recall=0.800  F1=0.235  bal_acc=0.771  AUC=0.844
SMOTE          recall=0.800  F1=0.238  bal_acc=0.773  AUC=0.845


La lectura es muy ilustrativa: el modelo base tiene un AUC alto (0,84) pero un recall de apenas 0,02, es decir, no detecta prácticamente ningún ictus. Tras equilibrar (con class_weight o con SMOTE), el recall sube al 0,80: el modelo pasa a identificar la mayoría de los casos de interés. Obsérvese que la accuracy o el AUC apenas cambian, lo que confirma por qué no bastan como métricas en problemas desequilibrados.

Para una comparación más fiable, en lugar de una única partición conviene usar validación cruzada estratificada sobre el ImbPipeline (con `cross_val_score` y una métrica como `recall`, `f1` o `balanced_accuracy`), de modo que el remuestreo se rehaga en cada pliegue.

**Buenas prácticas**

* Equilibra solo el entrenamiento; mantén el test (y cada pliegue de validación) con la proporción real de clases.
* Integra el remuestreo en un Pipeline de imblearn para que no se aplique al predecir y para que, en validación cruzada, se rehaga por pliegue.
* SMOTE y NearMiss necesitan datos numéricos y sin nulos: preprocesa antes (imputación, codificación, escalado), o usa SMOTENC si hay variables categóricas.
* Prueba primero class_weight='balanced': es simple y a menudo igual de eficaz que el remuestreo, sin generar datos sintéticos ni descartar información.
* No persigas un equilibrio perfecto como fin en sí mismo: el objetivo es mejorar la detección de la clase de interés, medida con métricas adecuadas (recall, F1, balanced accuracy, PR-AUC).

# <font color="steelblue">6. Referencias y enlaces de interés</font>

Los recursos utilizados para elaborar este módulo de aprendizaje, y que puedes utilizar para ampliar información sobre *Machine Learning* con *Scikit-Learn*, han sido:


* Scikit-learn (n.d.). An introduction to machine learning with scikit-learn. Retrieved January 10, 2024, from https://scikit-learn.org/stable/tutorial/basic/tutorial.html.


* Ander Fernández (2021). Tutorial Sklearn Python. Retrieved January 10, 2024, from https://anderfernandez.com/blog/tutorial-sklearn-machine-learning-python/.


* Aurélien Géron (2019). Hands-On Machine Learning with Scikit-Learn, Keras, and Tensorflow: Concepts, Tools, and Techniques to Build Intelligent Systems. O’Reilly Media.

* Peters Morgan (2018). Data Analysis From Scratch With Python: Beginner Guide using Python, Pandas, NumPy, Scikit-Learn, IPython, TensorFlow and Matplotlib. AI Sciences LLC.

* Sebastian Raschka, Vahid Mirjalili (2017). Python Machine Learning: Machine Learning and Deep Learning with Python, scikit-learn, and TensorFlow. Packt Publishing.

* Julian Avila (2017). Scikit-Learn Cookbook: Over 80 Recipes for Machine Learning in Python With Scikit-Learn. Packt Publishing.





